# Automatic Differentiation: Hands-On with Clad

[**Clad**](https://github.com/vgvassilev/clad) is a source-to-source automatic-differentiation (AD) plugin for the Clang/LLVM compiler. Given a C++ function, it *generates the C++ source of its derivative* at compile time (here, at ROOT/Cling JIT time) which is then compiled like any other code.
The derivative is therefore **exact** (not finite differences) and as fast as hand-written code.

In this notebook, we'll try out **Clad** through the [C++ interpreter in ROOT, called Cling](https://root.cern/manual/cling/), steered from the [ROOT Python Interface](https://root.cern/manual/python/).

This notebook is a hands-on tour in five sections:

1. **Clad fundamentals**: forward vs. reverse mode, inspecting the generated code, driving Clad from Python, and Hessians.
2. **When branches lie to the differentiator**: a discontinuity trap, and how to detect and fix it.
3. **Jacobians, jvp and vjp** vector-valued outputs, and the forward/reverse duality that maps onto JAX.
4. **Differentiable detector design**: putting a gradient to work in an optimizer.
5. **Measuring the Z mass from real CMS open data**: a binned likelihood fit, with gradient *and* Hessian from one function.

We write the C++ with ROOT's `%%cpp` cell magic: `--declare` *defines* functions; plain `%%cpp` runs the `clad::*` call that *triggers code generation*. The generated function then appears as a normal `ROOT.<name>`, callable from Python.

## Setup

**Prerequisites.** A recent ROOT build (Clad is bundled and enabled by default) plus
`numpy`, `matplotlib`, and `iminuit`:

```
mamba install -c conda-forge root numpy matplotlib iminuit
```

Importing `ROOT` in a Jupyter kernel registers the `%%cpp` cell magic used throughout.

> **If you hit `error: redefinition of '<name>'`:** Cling does not allow redefining a
> symbol, so re-running a `%%cpp --declare` cell fails. Recovery: **Kernel → Restart**,
> then re-run the cells above yours.


In [ ]:
import numpy as np
import ROOT
from iminuit import Minuit
import matplotlib.pyplot as plt

# warm up the Cling + Clad JIT (a few seconds) so later cells are snappy
ROOT.gInterpreter.Declare("#include <Math/CladDerivator.h>")
ROOT.gInterpreter.Declare("double _warm(double z){ return z*z; }")
ROOT.gInterpreter.ProcessLine("clad::gradient(_warm);")
print("ROOT", ROOT.gROOT.GetVersion())

## 1. Clad fundamentals

Three entry points, each taking a function and returning a callable:

| call | mode | gives |
|------|------|-------|
| `clad::differentiate(f, "x")` | forward | one directional derivative |
| `clad::gradient(f)` | reverse | all first partials at once |
| `clad::hessian(f, "x,y")` | second order | the full Hessian |

Our running example is $f(x, y) = x^2 y + \sin x$, with
$\partial f/\partial x = 2xy + \cos x$, $\partial f/\partial y = x^2$, and
$\partial^2 f/\partial x^2 = 2y - \sin x$, $\partial^2 f/\partial x\,\partial y = 2x$,
$\partial^2 f/\partial y^2 = 0$.

In [ ]:
%%cpp --declare

double f(double x, double y)
{
   return x * x * y + std::sin(x);
}


In [ ]:
x = 1.3
y = 2.0

### 1.1 Forward mode: `clad::differentiate`

Forward mode pushes one input's perturbation through the computation: one generated
function per input, each call returning that partial derivative. Cost scales with the
number of **inputs**. Good for few inputs, many outputs.


In [ ]:
%%cpp
clad::differentiate(f, "x"); // -> f_darg0
clad::differentiate(f, "y"); // -> f_darg1

In [ ]:
dfdx = ROOT.f_darg0(x, y)   # forward mode RETURNS the derivative directly
dfdy = ROOT.f_darg1(x, y)
print(f"df/dx = {dfdx:.6f}   (analytic {2*x*y + np.cos(x):.6f})")
print(f"df/dy = {dfdy:.6f}   (analytic {x*x:.6f})")

### 1.2 Reverse mode: `clad::gradient`

Reverse mode propagates the output's sensitivity backwards: **one call gives all partials
at once**. Cost scales with the number of **outputs**. The mode for fitting/ML, where one
scalar loss depends on many parameters. The gradient comes back through trailing output
arguments.


In [ ]:
%%cpp
clad::gradient(f); // -> f_grad

In [ ]:
# gradients come back through trailing double* args -> pass length-1 numpy arrays
gx = np.zeros(1)
gy = np.zeros(1)
ROOT.f_grad(x, y, gx, gy)
print(f"grad f = ({gx[0]:.6f}, {gy[0]:.6f})   "
      f"(analytic ({2*x*y + np.cos(x):.6f}, {x*x:.6f}))")

### 1.3 Inspecting the generated code with `.dump()`

Each entry point returns an object whose `.dump()` prints the generated source in plain C++.
You could paste into a codebase that does not link Clad at all.


In [ ]:
%%cpp
{
   auto d = clad::differentiate(f, "x");
   d.dump();
}


In [ ]:
%%cpp
{
   auto g = clad::gradient(f);
   g.dump();
}


### 1.4 From Python: array parameters

Real models pack parameters into a `double*`; from Python, pass one numpy array for the
parameters and one to receive the gradient. Two rules that bite: **contiguous float64**
arrays, and **zero the gradient array** before each call, as Clad *accumulates* into it.


In [ ]:
%%cpp --declare

// same f as before, parameters packed into an array
double g(double *p)
{
   return p[0] * p[0] * p[1] + std::sin(p[0]);
}


In [ ]:
%%cpp
clad::gradient(g, "p"); // -> g_grad


In [ ]:
p = np.array([x, y])
grad = np.zeros(2)                 # must be zeroed: Clad adds into it
ROOT.g_grad(p, grad)               # generated name: g_grad
print("params p  =", p)
print("grad g(p) =", grad, "  (same numbers, array-valued)")

### 1.5 Hessians with `clad::hessian`

`clad::hessian` fills a flattened $n \times n$ buffer of second derivatives
(forward-over-reverse under the hood); reshape to $(n, n)$. The headline use is
uncertainties: for a negative log-likelihood, the parameter covariance is the inverse
Hessian at the minimum, $C = H^{-1}$.


In [ ]:
%%cpp
clad::hessian(f, "x,y"); // -> f_hessian


In [ ]:
H = np.zeros(4)
ROOT.f_hessian(x, y, H)
H = H.reshape(2, 2)
print("Hessian =")
print(H)
print("analytic = [[2y - sin(x), 2x], [2x, 0]] =",
      [[round(2*y - np.sin(x), 4), 2*x], [2*x, 0.0]])

**Exercise.** Your "Hello World" with Clad: `cube(x) = x*x*x` has derivative $3x^2$,
so $12$ at $x = 2$. Use Clad to generate the derivative and check it.

*(Exercises drive Clad from Python via `ROOT.gInterpreter`, which is exactly what `%%cpp` does
under the hood, so everything fits in one cell.)*


In [ ]:
# EXERCISE: differentiate cube(x) = x*x*x with Clad and check cube'(2) == 12.
#   1. declare:         ROOT.gInterpreter.Declare("double cube(double x){ return x*x*x; }")
#   2. differentiate:   ROOT.gInterpreter.ProcessLine("clad::gradient(cube);")
#   3. call cube_grad(2.0, out) with a length-1 numpy array 'out'; print out[0]



**Solution** *(Collapsed below, expand after giving it a try! A plain "Run All" still executes it)*.


In [ ]:
ROOT.gInterpreter.Declare("double cube(double x){ return x*x*x; }")
ROOT.gInterpreter.ProcessLine("clad::gradient(cube);")
d = np.zeros(1)
ROOT.cube_grad(2.0, d)
print("cube'(2) =", d[0], " (analytic 12)")


## 2. When branches lie to the differentiator

Everywhere else in this notebook AD "just works". Here is the sharp edge, in one sentence:

> Automatic differentiation differentiates the **code path you actually execute**, which
> is not always the **math you meant**.

Clad is faithful: it returns the exact derivative of the branch that ran. The trouble is
that an innocent-looking `if` can make that derivative disagree with the function you
meant, while the *value* looks perfectly fine.


In [ ]:
%%cpp --declare

// f(x) = x*x - 3x, so f(0) = 0
double para_plain(double x)
{
   return x * x - 3.0 * x;
}
double para_opt(double x)
{
   if (x == 0.0)
      return 0.0; // "optimization": we KNOW f(0) = 0
   return x * x - 3.0 * x;
}


In [ ]:
%%cpp
clad::gradient(para_plain);
clad::gradient(para_opt);


In [ ]:
def clad_grad(name, xv):
    # scalar argument -> the generated name is just <name>_grad
    d = np.zeros(1)
    getattr(ROOT, name + "_grad")(xv, d)
    return d[0]
def fin_diff(name, xv, h=1e-6):
    fnc = getattr(ROOT, name)
    return (fnc(xv + h) - fnc(xv - h)) / (2*h)
print("helpers ready")

### The fast path: correct value, silently wrong derivative

$f(x) = x(x-3)$ has $f(0) = 0$, so a shortcut `if (x == 0) return 0;` looks harmless. The
value is right everywhere and the derivative matches the honest version at *every* point
except $x = 0$, where AD reports $0$ instead of the true $f'(0) = -3$. A unit test at any
random $x$ passes; the one point you special-cased is the one AD gets wrong.

In [ ]:
print(f"{'x':>6} {'value':>10} {'plain f_x':>11} {'opt f_x':>10} {'finite diff':>12}")
for xv in [-1.0, 0.0, 1.0, 2.0]:
    print(f"{xv:6.1f} {ROOT.para_opt(xv):10.3f} {clad_grad('para_plain', xv):11.3f} "
          f"{clad_grad('para_opt', xv):10.3f} {fin_diff('para_opt', xv):12.3f}")

### Detect, and fix

A cheap habit: spot-check every generated gradient against a central finite difference
with a not-too-small step. The last column above flags the trap immediately. The real fix
is to remove the discontinuity, not to differentiate around it: here the fast path is
simply needless. The same medicine returns in section 5: the binned model is evaluated at
fixed bin centers, keeping the parameters out of branch conditions.


**Exercise.** `para_opt` returns the right *value*, but its AD derivative at $x = 0$ is $0$
instead of the true $-3$. Write a fixed version with the same values but a correct
derivative everywhere, and check that its gradient at $x = 0$ is $-3$.


In [ ]:
# EXERCISE: write 'para_fixed' -- same values as para_opt, correct derivative everywhere --
# differentiate it, and verify para_fixed'(0) == -3.
#   Hint: f(0) = 0 already falls out of x*x - 3*x, so just drop the special case.
#   The clad_grad(name, x) helper from above is handy.



**Solution** *(Collapsed below, expand after giving it a try! A plain "Run All" still executes it)*.


In [ ]:
ROOT.gInterpreter.Declare("double para_fixed(double x){ return x*x - 3.0*x; }")
ROOT.gInterpreter.ProcessLine("clad::gradient(para_fixed);")
print("para_fixed(0)  =", ROOT.para_fixed(0.0), " (same value as para_opt)")
print("para_fixed'(0) =", clad_grad("para_fixed", 0.0), " (fixed: -3, not 0)")


## 3. Jacobians, jvp and vjp

So far, forward (`differentiate`) and reverse (`gradient`) acted on a **scalar** output.
For a **vector** output the derivative is a full **Jacobian**, and the two dual ways of
touching it get the names you may know from JAX:

- forward mode $\to$ **jvp** ("push a tangent forward", $J\,v$)
- reverse mode $\to$ **vjp** ("pull a cotangent back", $J^\top w$)

The physics example in this section: a track measured as $(p_T, \eta, \phi)$ with a covariance matrix needs its
momentum in Cartesian coordinates,
$$p_x = p_T\cos\phi, \qquad p_y = p_T\sin\phi, \qquad p_z = p_T\sinh\eta,$$
and moving the covariance needs the Jacobian: $C' = J\,C\,J^\top$.

| Clad | gives | JAX |
|------|-------|-----|
| `clad::jacobian(f)` | full $J$, **forward** | `jax.jacfwd` |
| `clad::differentiate(f, "x_k")` | column $k = J e_k$ | `jax.jvp` |
| `clad::gradient(f_component_j)` | row $j = e_j^\top J$ | `jax.vjp` |
| (stack all component gradients) | reverse-mode $J$ | `jax.jacrev` |

`clad::jacobian` uses vectorized **forward** mode. All input tangents pushed through in
one sweep (the `jacfwd` analogue). Every scalar `clad::gradient` in this notebook is a row
of some Jacobian: a vjp with cotangent $w = 1$.

One convention: an **output** array parameter carries a `_clad_out_` prefix so Clad treats
it as the output rather than another independent variable; the derivative matrix is then a
clean $n_\text{out}\times n_\text{in}$.


In [ ]:
%%cpp --declare

// in = (pT, eta, phi) -> out = (px, py, pz); the _clad_out_ prefix marks the output array.
void trk(double *in, double *_clad_out_out)
{
   double pT = in[0], eta = in[1], phi = in[2];
   _clad_out_out[0] = pT * std::cos(phi);
   _clad_out_out[1] = pT * std::sin(phi);
   _clad_out_out[2] = pT * std::sinh(eta);
}
// Shim: clad::jacobian returns a functor; .execute takes inputs, outputs, and one
// caller-allocated clad::matrix<double>* per array parameter. d_out is the 3x3 Jacobian
// (rows = outputs, cols = inputs); copy it out flat.
void trk_jacobian(double *in, double *Jflat)
{
   auto jac = clad::jacobian(trk, "in");
   double out[3];
   clad::matrix<double> d_in(3, 3), d_out(3, 3);
   jac.execute(in, out, &d_in, &d_out);
   for (int i = 0; i < 3; ++i)
      for (int k = 0; k < 3; ++k)
         Jflat[i * 3 + k] = d_out[i][k];
}


In [ ]:
pT, eta, phi = 10.0, 0.5, 0.3
xtrk = np.array([pT, eta, phi])
analytic = np.array([[np.cos(phi),  0.0,              -pT*np.sin(phi)],
                     [np.sin(phi),  0.0,               pT*np.cos(phi)],
                     [np.sinh(eta), pT*np.cosh(eta),   0.0]])
print("evaluation point (pT, eta, phi) =", xtrk)


### 3.1 The full Jacobian in one call with `clad::jacobian`

In [ ]:
J = np.zeros(9)
ROOT.trk_jacobian(xtrk, J)
J = J.reshape(3, 3)
print("J = d(px,py,pz)/d(pT,eta,phi) =")
print(np.array2string(J, precision=5))
assert np.allclose(J, analytic), "clad::jacobian disagrees with the analytic Jacobian"
print("matches analytic: True")

### 3.2 Error propagation example with covariance, $C' = J\,C\,J^\top$

The matrix generalization of the scalar error propagation $\sigma^2 = g^\top C\,g$. We
check the propagated covariance is symmetric and positive-definite.

The same derivative-based propagation has a second life inside Clad itself: its
[floating-point error estimation](https://clad.readthedocs.io/en/latest/user/UsingClad.html#error-estimation)
mode (`clad::estimate_error`) uses the reverse-mode derivatives to track how *rounding*
errors accumulate through your code.


In [ ]:
sigma = np.array([0.15, 0.0020, 0.0015])          # sigma_pT [GeV], sigma_eta, sigma_phi
corr = np.array([[1.0,  0.10, -0.20],
                 [0.10, 1.0,   0.05],
                 [-0.20, 0.05, 1.0]])
C_in = np.outer(sigma, sigma) * corr
Cp = J @ C_in @ J.T
evals = np.linalg.eigvalsh(Cp)
print("C' (px,py,pz) [GeV^2] =")
print(np.array2string(Cp, precision=5))
print("symmetric:", np.allclose(Cp, Cp.T),
      "  positive-definite:", bool(np.all(evals > 0)),
      "  eigs", np.array2string(evals, formatter={'float_kind': lambda z: f'{z:.1e}'}))
print("sigma(px, py, pz) =", np.array2string(np.sqrt(np.diag(Cp)), precision=4), "GeV")

## 4. Differentiable detector design

An actual design question: **how thick should the absorber plates be?** The workflow: *a
C++ model with a loop $\to$ an objective $\to$ Clad gradient $\to$ optimizer*.

A sampling calorimeter is a sandwich of dense **absorber** plates (lead) and thin **active**
layers (scintillator, which produces the signal). An electron of energy $E_0$ showers
through the stack, depositing on average $dE/dt \propto (bt)^{a-1} e^{-bt}$ along its
depth ($t$ in radiation lengths). With the layer count fixed, the absorber thickness sets
the energy resolution through two competing effects:

- **too thin**: the stack is too shallow and the shower **leaks** out the back;
- **too thick**: the sampling is coarse and **sampling fluctuations** grow ($\propto \sqrt{d_\text{abs}}$).

The toy objective $\sigma_E/E = \sqrt{\sigma_\text{samp}^2 + \sigma_\text{leak}^2}$
captures both, and we're defining it as a parametrized formula, similar to what you would do in **fast simulation** (à la **Delphes**): detector
response written as smooth formulas of geometry and energy. This is *not* differentiable "full simulation": there, $\sigma_E$ would
be estimated from many stochastic showers, and differentiating *that* is a research field
of its own.

In [ ]:
%%cpp --declare

// Toy model: integrates the PARAMETRIZED AVERAGE longitudinal shower profile
// dE/dt ~ (bt)^(a-1) e^(-bt), t in radiation lengths -- no stochastic shower simulation here.
double resolution(double d_abs, double E0, int n_layers)
{
   const double X0_abs = 0.56, X0_act = 42.0, d_act = 0.5, b = 0.5, Ec = 0.0074;
   double a = 1.0 + b * (std::log(E0 / Ec) - 0.5); // profile shape; shower max at t = ln(E0/Ec) - 0.5
   double t = 0.0, e_in = 0.0;
   for (int i = 0; i < n_layers; ++i) { // mean deposit inside the stack: one midpoint-rule step per slab
      double dt = d_abs / X0_abs;       // absorber slab, in radiation lengths
      double tc = t + 0.5 * dt;
      e_in += std::pow(b * tc, a - 1.0) * std::exp(-b * tc) * dt;
      t += dt;
      dt = d_act / X0_act; // active slab
      tc = t + 0.5 * dt;
      e_in += std::pow(b * tc, a - 1.0) * std::exp(-b * tc) * dt;
      t += dt;
   }
   double e_out = 0.0;
   for (int i = 0; i < 200; ++i) { // mean tail leaking out the back: integrate profile to ~infinity
      double tc = t + 0.25;
      e_out += std::pow(b * tc, a - 1.0) * std::exp(-b * tc) * 0.5;
      t += 0.5;
   }
   double f_leak = e_out / (e_in + e_out);
   double sig_samp = 0.11 * std::sqrt(d_abs / X0_abs) / std::sqrt(E0); // Poisson: sigma/E ~ 1/sqrt(N crossings)
   double sig_leak = f_leak;                                           // exp. tail: spread of leakage ~ its mean
   return std::sqrt(sig_samp * sig_samp + sig_leak * sig_leak);
}

In [ ]:
%%cpp
clad::differentiate(resolution, "d_abs"); // one design parameter -> forward mode (section 1.1)

In [ ]:
print("energy resolution vs absorber thickness (10 GeV e-, 40 layers):")
for d in [0.1, 0.2, 0.3, 0.5, 1.0, 1.5]:
    print(f"  d_abs = {d:.2f} cm   ->   sigma_E/E = {ROOT.resolution(d, 10.0, 40):.3%}")

### The gradient, and the design optimization

The table shows the valley: leakage punishes thin absorbers steeply, coarse sampling
punishes thick ones gently. `clad::differentiate` gives the exact geometry gradient
(`resolution_darg0` returns it directly), and gradient descent finds the optimum.


In [ ]:
d_abs = 1.2                        # start on the thick side
lr, cap = 1.0, 0.05                # capped step: to not jump out of the valid region
for i in range(150):
    g = ROOT.resolution_darg0(d_abs, 10.0, 40)
    d_abs -= max(min(lr*g, cap), -cap)
    if d_abs < 0.05: d_abs = 0.05  # positivity clamp
    if i % 25 == 0 or i == 149:
        print(f"iter {i:03d} | d_abs={d_abs:.4f} cm | sigma_E/E={ROOT.resolution(d_abs, 10.0, 40):.3%}")
print(f"optimal absorber thickness: {d_abs:.3f} cm")

In [ ]:
ds = np.linspace(0.15, 1.5, 200)
rs = np.array([ROOT.resolution(d, 10.0, 40) for d in ds])
plt.figure(figsize=(6, 4))
plt.plot(ds, rs*100, label="simulated resolution")
plt.plot([d_abs], [ROOT.resolution(d_abs, 10.0, 40)*100], "o", color="crimson",
         label=f"optimum ({d_abs:.3f} cm)")
plt.xlabel("absorber thickness d_abs [cm]"); plt.ylabel("sigma_E/E [%]")
plt.legend(); plt.title("differentiable detector design: resolution vs geometry"); plt.show()


## 5. Measuring the Z mass from real CMS open data

The finale: **real data**! A proper likelihood, a real minimizer
(Minuit), and Hessian-based uncertainties. We're looking at an actual measurement: the Z
boson mass, and how many Z bosons are in the sample.

The signal is the relativistic **Breit-Wigner**

$$S(m) = \frac{1}{(m^2 - M^2)^2 + M^2\Gamma^2},$$

on a falling exponential background $B(m) = e^{-\lambda m}$. The usual pain of a mass fit
is the **normalization** integral (an $\arctan$ for the Breit-Wigner). In the following **binned
extended-Poisson** we'll sidestep this by approximating the integral by a *sum over bins*:

$$\nu_i = N_s\,\frac{S(m_i)}{\sum_j S(m_j)} + N_b\,\frac{B(m_i)}{\sum_j B(m_j)},$$

at fixed bin centers $m_i$, with extended NLL $\sum_i (\nu_i - n_i \ln \nu_i)$. No
special functions, nothing exotic for Clad. **Note**: the normalization
$\sum_j S(m_j;\,M,\Gamma)$ depends on the shape parameters, so
$\partial(\mathrm{NLL})/\partial M$ has a term flowing through it, which hand-coded
gradients routinely forget.

The data: 40 bins of 1 GeV over 70-110 GeV of opposite-sign dimuon invariant masses from
the CMS Run2011A DoubleMu dataset ([CERN Open Data record 545](https://opendata.cern.ch/record/545)), pre-binned so there is no
data loading.


In [ ]:
COUNTS = np.array([
    34, 30, 42, 30, 35, 53, 42, 43, 42, 47, 62, 59, 70, 94, 96, 128,
    171, 263, 422, 675, 806, 873, 595, 314, 228, 107, 80, 54, 46, 31,
    27, 21, 12, 14, 13, 19, 11, 7, 13, 7], dtype=float)
M_LO, M_HI, N_BINS = 70.0, 110.0, 40
BIN_W = (M_HI - M_LO) / N_BINS
CENTERS = M_LO + (np.arange(N_BINS) + 0.5) * BIN_W
PDG_MZ = 91.1876

print(f"{int(COUNTS.sum())} events; tallest bin {CENTERS[COUNTS.argmax()]:.1f} GeV with {int(COUNTS.max())} events")
plt.figure(figsize=(6, 4))
plt.bar(CENTERS, COUNTS, width=BIN_W*0.9, color="steelblue")
plt.xlabel("dimuon invariant mass [GeV]"); plt.ylabel("events / GeV")
plt.title("CMS open data: dimuon spectrum"); plt.show()

### Where the counts come from (reproducing them)

`COUNTS` is real CMS data, pre-binned so the notebook needs no network or file access: the
**CMS Run2011A DoubleMu** dataset on the [CERN Open Data Portal, record 545](https://opendata.cern.ch/record/545),
whose `Dimuon_DoubleMu.csv` carries a precomputed dimuon invariant-mass column `M`
(475k events). The counts were produced once with:

```python
import numpy as np

# Dimuon_DoubleMu.csv from record 545, e.g. via XRootD:
#   root://eospublic.cern.ch//eos/opendata/cms/Run2011A/DoubleMu/CSV/12Oct2013-v1/Dimuon_DoubleMu.csv

d = np.genfromtxt("Dimuon_DoubleMu.csv", delimiter=",", names=True)
opp = d["Q1"] * d["Q2"] < 0                          # opposite-sign muons only
COUNTS, edges = np.histogram(d["M"][opp], bins=40, range=(70, 110))
```

Change `bins`/`range` and everything downstream (fit, Hessian errors) adapts.


### 5.1 The model and likelihood

The parameters `[N_s, N_b, M, Gamma, lambda]` are packed into a `double*` (the array shape
from section 1), and the bin counts come in as a `const double*`. The model is written
with small line-shape functions, and an NLL that calls them in a loop. Clad
differentiates through **nested function calls** without any special treatment. From the
one NLL we generate both the gradient (for the fit) and the Hessian (for the error bars).


In [ ]:
%%cpp --declare

const int NBINS = 40;
const double MLO = 70.0;
const double MHI = 110.0;
// line shapes, bare (the NLL normalizes them by their sum over bins)
double breit_wigner(double m, double M, double G)
{
   double d = m * m - M * M;
   return 1.0 / (d * d + M * M * G * G);
}
double background(double m, double lam)
{
   return std::exp(-lam * m);
}
double nll(double *p, double const *n)
{
   const double binw = (MHI - MLO) / NBINS;
   double Ns = p[0], Nb = p[1], M = p[2], G = p[3], lam = p[4];
   double sumS = 0.0, sumB = 0.0;
   for (int i = 0; i < NBINS; ++i) {
      double m = MLO + (i + 0.5) * binw;
      sumS += breit_wigner(m, M, G);
      sumB += background(m, lam);
   }
   double val = 0.0;
   for (int i = 0; i < NBINS; ++i) {
      double m = MLO + (i + 0.5) * binw;
      double nu = Ns * breit_wigner(m, M, G) / sumS + Nb * background(m, lam) / sumB;
      val += nu - n[i] * std::log(nu);
   }
   return val;
}


**Naming of the generated functions.** Reverse mode is `<f>_grad`, the Hessian
`<f>_hessian`, forward mode `<f>_darg<k>` (one per input `k`). One wrinkle: when the
differentiated argument is an **array** *and* the function takes further arguments (as
`nll` does, taking the parameters **and** the bin counts), Clad appends an index:
`nll_grad_0` and `nll_hessian_0`. Plain-scalar functions and a lone array argument keep
the bare name.


In [ ]:
%%cpp
clad::gradient(nll, "p");     // -> nll_grad_0
clad::hessian(nll, "p[0:4]"); // -> nll_hessian_0

### 5.2 The fit with Minuit driven by the Clad gradient

First spot-check the fresh gradient against a central finite difference (the habit from
section 2), then fit. No parameter rescaling: the scales differ wildly (yields ~1e3,
$m_Z$ ~ 90, $\lambda$ ~ 0.03), but Minuit keeps a per-parameter step size, so it fits the
**natural** parameters directly. The C++ NLL is the cost; the Clad gradient is the
Jacobian. The extended fit's total prediction matches the observed count.


In [ ]:
# spot-check the fresh gradient against finite differences
p0 = np.array([4300.0, 1400.0, 91.0, 3.0, 0.03])
ga = np.zeros(5); ROOT.nll_grad_0(p0, COUNTS, ga)
gn = np.zeros(5)
for i in range(5):
    h = 1e-6 * max(1.0, abs(p0[i]))
    pp = p0.copy(); pp[i] += h
    pm = p0.copy(); pm[i] -= h
    gn[i] = (ROOT.nll(pp, COUNTS) - ROOT.nll(pm, COUNTS)) / (2*h)
print("gradient check vs finite differences:", "PASS" if np.allclose(ga, gn, rtol=1e-3) else "FAIL")


In [ ]:
def cost(par):
    return ROOT.nll(np.ascontiguousarray(par, dtype=float), COUNTS)
def grad(par):
    d = np.zeros(5)
    ROOT.nll_grad_0(np.ascontiguousarray(par, dtype=float), COUNTS, d); return d

m = Minuit(cost, p0, grad=grad, name=["N_s", "N_b", "M", "Gamma", "lambda"])
m.errordef = Minuit.LIKELIHOOD    # plain NLL: 1-sigma at delta-NLL = 0.5
m.limits = [(1, None), (1, None), (85, 97), (0.5, 12), (1e-3, 0.2)]
m.migrad()
fit = np.array(m.values); Ns, Nb, MZ, Gam, lam = fit
print(f"valid minimum: {m.valid}   ({m.nfcn} cost calls, {m.ngrad} gradient calls, NLL = {m.fval:.2f})")
print(f"  N_s = {Ns:8.1f}   N_b = {Nb:8.1f}   (sum {Ns+Nb:.0f}, data {COUNTS.sum():.0f})")
print(f"  m_Z = {MZ:.4f} GeV   Gamma = {Gam:.4f} GeV   lambda = {lam:.4f} /GeV")


### 5.3 Uncertainties by inverting the AD Hessian

For a plain NLL the covariance is exactly the inverse Hessian: $C = H^{-1}$ at the
minimum; 1-sigma errors are the square roots of its diagonal.

**Exercise.** `clad::hessian` already generated `nll_hessian_0`. Fill the 5×5 Hessian at
the fitted minimum, invert it, and read off the 1-sigma error on $m_Z$ (parameter
index 2). Store the error vector as `err` (the plot in the next section uses it). Cross-check against
Minuit's numeric `m.hesse()` errors.


In [ ]:
# EXERCISE: uncertainties from the Clad Hessian.
#   1. H = np.zeros(25); ROOT.nll_hessian_0(np.ascontiguousarray(fit), COUNTS, H)
#   2. err = np.sqrt(np.diag(np.linalg.inv(H.reshape(5, 5))))
#   3. print m_Z = fit[2] +/- err[2]; cross-check with m.hesse(); m.errors
# (define err -- the 5.4 plot uses it)



**Solution** *(Collapsed below, expand after giving it a try! A plain "Run All" still executes it)*.


In [ ]:
H = np.zeros(25)
ROOT.nll_hessian_0(np.ascontiguousarray(fit), COUNTS, H)
cov = np.linalg.inv(H.reshape(5, 5))
err = np.sqrt(np.diag(cov))
m.hesse()   # Minuit's numeric Hesse, as an independent cross-check

print(f"{'param':6} {'value':>12} {'Clad Hessian':>14} {'Minuit Hesse':>14}")
for name, vv, ee, me in zip(["N_s", "N_b", "M", "Gamma", "lambda"], fit, err, np.array(m.errors)):
    print(f"{name:6} {vv:12.4f} {ee:14.4f} {me:14.4f}")
print()
print(f"RESULT:  m_Z = {MZ:.3f} +/- {err[2]:.3f} GeV (stat.)   PDG: {PDG_MZ:.3f} GeV")
print(f"         N_Z = {Ns:.0f} +/- {err[0]:.0f} reconstructed Z -> mu mu decays")


### 5.4 The fit


In [ ]:
def bw_bkg(par):
    Ns, Nb, M, G, lam = par
    d = CENTERS**2 - M**2
    S = 1.0/(d*d + M*M*G*G)
    B = np.exp(-lam*CENTERS)
    return Ns*S/S.sum(), Nb*B/B.sum()

sig, bkg = bw_bkg(fit)
plt.figure(figsize=(7.5, 5))
plt.errorbar(CENTERS, COUNTS, yerr=np.sqrt(COUNTS), fmt="o", ms=4, color="black", label="CMS open data", zorder=5)
plt.plot(CENTERS, sig+bkg, "-", color="crimson", lw=2, label=f"Breit-Wigner fit (m_Z={MZ:.2f})")
plt.plot(CENTERS, bkg, ":", color="steelblue", label="background")
plt.axvline(PDG_MZ, color="gray", ls=":", alpha=0.8)
plt.xlabel("dimuon invariant mass [GeV]"); plt.ylabel("events / GeV")
plt.title(f"Z peak from CMS open data: m_Z = {MZ:.2f} +/- {err[2]:.2f} GeV (stat.)")
plt.legend(); plt.show()


## Conclusions

The same tool did a lot of different work, all from ordinary C++:

- **exact gradients** for optimization, forward vs. reverse mode and when each wins;
- **Jacobians** and the **jvp / vjp** duality that carries straight over to JAX;
- **Hessians** for uncertainties ($C = H^{-1}$), cross-checked against Minuit;
- and the caveat: **AD differentiates the code you wrote, not the math you meant**! Keep
  models smooth, and finite-difference-check fresh gradients.

We ended with a real measurement: the Z mass and yield from CMS open data, with the
statistical error straight from the inverse of the Clad Hessian.

The short demo here was just the tip of the iceberg!
Automatic Differentiation (AD) can be used for many applications, and there are many tools that enable AD for different programming languages in various ways.

For Clad, the compelling story is: you do not rewrite your physics into a tensor
framework to get gradients. **If you can write it in C++, you can differentiate it**,
and plug the result straight into an optimizer, a sampler, or an error propagation.
